# Lecture 7.6: Subagent Limits, Hooks & Cost

**Section 07: Subagents: Delegation & Multi-Agent Workflows**

Once `Agent` is in `allowed_tools`, Claude decides on its own when to start a subagent and how many to start. The Bike Repair Cooperative (the co-op) runs three specialist subagents: a parts clerk, a triage mechanic and a shift planner. In this notebook you take back control over them in four ways:

| Tool | What it controls | Precise or blunt? |
|---|---|---|
| `SubagentStart` / `SubagentStop` hooks | A log of every specialist that starts and stops, with its transcript | Precise: you see each subagent by name and ID |
| A `PreToolUse` hook that reads `agent_type` | Blocks one specific specialist from one specific tool | Precise: every other agent is unaffected |
| `CLAUDE_CODE_MAX_CONCURRENT_SUBAGENTS` and `CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH` in `env` | How many subagents run at once, and how deeply they nest | Blunt: applies to every subagent |
| `max_budget_usd` | How much the whole query may spend, subagents included | Blunt: one number for the whole tree |

The last cell shows where subagent cost appears in the `ResultMessage`.

Agent wording and token counts change from run to run. The ground truth in this notebook is the log file on disk, which files exist, and the timeline of the stream.

## Cell 1: Install the Claude Agent SDK

This cell installs the Claude Agent SDK, the Python package that runs the agent loop and the specialists you meet in this notebook. The package bundles the Claude Code CLI that does the actual work, so nothing else needs installing.

The version is pinned so that every run behaves like the one in the video. The comment in the cell explains how to install the latest release instead. If the package is already installed in this Colab session, pip notices and finishes quickly.

In [33]:
# We pin the Claude Agent SDK to a specific version so every example runs
# exactly as shown in the course. To use the latest version, run:
# pip install -U claude-agent-sdk
# Or substitute your preferred version below. Newer versions may behave
# differently from the videos.
!pip install claude-agent-sdk==0.2.174 -q

## Cell 2: Set your API key from Colab Secrets

The SDK reads your Anthropic API key from the `ANTHROPIC_API_KEY` environment variable. In Colab, the safe place to keep the key is Colab Secrets, so it never appears in the notebook.

If you haven't added the secret yet:

1. Click the **key icon** in Colab's left sidebar.
2. Click **Add new secret**.
3. Name it exactly `ANTHROPIC_API_KEY`.
4. Paste your key into the value field. You create keys in the Claude Console at https://platform.claude.com/ under **API Keys**.
5. Switch **Notebook access** on for this notebook.

The cell reads the secret and copies it into the environment, where the SDK looks for it.

> **Running locally instead of Colab?** Set the variable in your terminal before starting Jupyter, for example `export ANTHROPIC_API_KEY=...`, and skip this cell.

In [34]:
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

## Cell 3: Choose the model

`MODEL_NAME` is the one place this notebook names a model. The main agent and all three specialists use it, so changing this variable changes the model across the entire notebook.

The course default is Claude Haiku 5.5. To try a different model, check the models page linked in the cell for current names.

In [35]:
# See latest models at: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL_NAME = "claude-haiku-5-5"

## Cell 4: Create the co-op files

This cell builds the co-op's paperwork under `/content/bikecoop/`. It is the same generator as the rest of this section, and its fixed random seed means you get the same files every time you run it.

| File | Contents | Built-in facts |
|---|---|---|
| `work_orders/WO-101.txt` to `WO-140.txt` | 40 repair tickets: customer, bike, issue, skill needed, priority, safety flag | Exactly **five** are `Priority: urgent` with `Safety issue: yes`. No other order is urgent or flagged. |
| `inventory.csv` | 15 parts with `on_hand` and `reorder_level` | Exactly **four** parts have `on_hand` below `reorder_level`. |
| `volunteers.txt` | 6 volunteers with their skills and shift days | Every skill an urgent order needs is covered by at least one volunteer. |

Those built-in facts are the ground truth for the whole notebook.

The cell also deletes the files that later cells create (`agent_log.jsonl`, `rota.txt`, `summary.txt` and any `briefing*.md`), so every check below starts from a clean folder.

In [36]:
import csv
import random
from pathlib import Path

random.seed(75)

BASE = Path("/content/bikecoop")
ORDERS = BASE / "work_orders"
ORDERS.mkdir(parents=True, exist_ok=True)
for old in (list(ORDERS.glob("*.txt")) + list(BASE.glob("briefing*.md"))
            + [BASE / n for n in ("agent_log.jsonl", "rota.txt", "summary.txt")]):
    old.unlink(missing_ok=True)

CUSTOMERS = ["Priya", "Tom", "Aiko", "Marcus", "Lena", "Omar", "Grace",
             "Ravi", "Sofia", "Kwame", "Hana", "Diego"]
BIKES = ["Commuter hybrid", "Road bike", "Mountain bike", "Cargo bike",
         "Folding bike", "Kids' bike"]

# The five urgent safety issues: (skill needed, issue)
URGENT_ISSUES = [
    ("brakes", "Rear brake cable frayed; lever pulls all the way to the bar"),
    ("wheels", "Front wheel quick release will not lock"),
    ("frame", "Crack spotted next to the seat tube weld"),
    ("brakes", "Hydraulic brake leaking fluid onto the front rotor"),
    ("drivetrain", "Chain snapped once on a hill and now skips under load"),
]
ROUTINE_ISSUES = [
    ("fit", "Saddle creaks when pedalling"),
    ("drivetrain", "Gears need re-indexing"),
    ("wheels", "Slow puncture in the rear tube"),
    ("accessories", "Bell missing"),
    ("electrical", "Rear light flickers"),
    ("fit", "Bar tape worn through"),
    ("accessories", "Kickstand loose"),
    ("wheels", "Spoke tension check"),
    ("brakes", "Brake pads squeal but stop fine"),
]

order_ids = [f"WO-{n}" for n in range(101, 141)]
urgent_ids = sorted(random.sample(order_ids, 5))
urgent_issue = dict(zip(urgent_ids, URGENT_ISSUES))

for i, oid in enumerate(order_ids):
    if oid in urgent_issue:
        skill, issue = urgent_issue[oid]
        priority, safety = "urgent", "yes"
    else:
        skill, issue = random.choice(ROUTINE_ISSUES)
        priority, safety = random.choice(["normal", "normal", "low"]), "no"
    (ORDERS / f"{oid}.txt").write_text(
        f"Order ID: {oid}\n"
        f"Received: 2026-11-{(i % 14) + 2:02d}\n"
        f"Customer: {random.choice(CUSTOMERS)}\n"
        f"Bike: {random.choice(BIKES)}\n"
        f"Issue: {issue}\n"
        f"Skill needed: {skill}\n"
        f"Priority: {priority}\n"
        f"Safety issue: {safety}\n"
    )

# 15 parts; exactly four are below their reorder level
PARTS = [
    ("brake cable", 2, 5), ("disc brake pads", 12, 6), ("rim brake pads", 9, 4),
    ("inner tube 700c", 3, 10), ("inner tube 26in", 14, 6), ("chain 9-speed", 1, 4),
    ("quick release skewer", 7, 3), ("hydraulic brake fluid", 1, 3), ("spokes", 60, 20),
    ("bar tape", 8, 4), ("saddle clamp", 6, 2), ("rear LED light", 10, 4),
    ("bell", 11, 5), ("tyre levers", 20, 8), ("cable housing", 15, 6),
]
with open(BASE / "inventory.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["part", "on_hand", "reorder_level"])
    writer.writerows(PARTS)

VOLUNTEERS = [
    ("Asha", "brakes, wheels", "Tue, Sat"),
    ("Ben", "drivetrain, fit", "Mon, Thu"),
    ("Carmen", "frame, brakes", "Wed, Sat"),
    ("Dev", "wheels, electrical", "Tue, Fri"),
    ("Elena", "drivetrain, frame", "Thu, Sun"),
    ("Farid", "accessories, fit", "Mon, Wed"),
]
(BASE / "volunteers.txt").write_text(
    "\n".join(f"{name} | skills: {skills} | shifts: {days}"
              for name, skills, days in VOLUNTEERS) + "\n"
)

print("Co-op files created under", BASE)

Co-op files created under /content/bikecoop


## Cell 5: Verify the files and print the ground truth

Before any agent runs, this cell shows you exactly what is on disk:

- the directory tree,
- one sample work order, so you know the ticket format,
- the full inventory and the volunteer list.

Then it reads the files back and computes the two answers every later cell checks against:

| Variable | How it is computed |
|---|---|
| `LOW_STOCK` | Parts whose `on_hand` is below `reorder_level` in `inventory.csv` |
| `URGENT_IDS` | Orders containing both `Priority: urgent` and `Safety issue: yes` |

It also lists which volunteers have the skill each urgent order needs, which confirms every urgent order can be staffed. These values come from the files, not from an agent, so they are the ground truth.

In [37]:
print("Directory tree:")
for path in sorted(BASE.rglob("*")):
    depth = len(path.relative_to(BASE).parts) - 1
    print("   " * depth + path.name)

print("\nSample work order (WO-101.txt):")
print((ORDERS / "WO-101.txt").read_text())
print("inventory.csv:")
print((BASE / "inventory.csv").read_text())
print("volunteers.txt:")
print((BASE / "volunteers.txt").read_text())

# Ground truth, computed from the files themselves
with open(BASE / "inventory.csv") as f:
    LOW_STOCK = [row["part"] for row in csv.DictReader(f)
                 if int(row["on_hand"]) < int(row["reorder_level"])]

URGENT_IDS, NEEDS = [], {}
for path in sorted(ORDERS.glob("*.txt")):
    text = path.read_text()
    if "Priority: urgent" in text and "Safety issue: yes" in text:
        URGENT_IDS.append(path.stem)
        NEEDS[path.stem] = text.split("Skill needed: ")[1].splitlines()[0]

print("GROUND TRUTH")
print(f"Low-stock parts ({len(LOW_STOCK)}):", ", ".join(LOW_STOCK))
print(f"Urgent safety orders ({len(URGENT_IDS)}):", ", ".join(URGENT_IDS))
for oid in URGENT_IDS:
    able = [v[0] for v in VOLUNTEERS if NEEDS[oid] in v[1]]
    print(f"  {oid} needs {NEEDS[oid]:<10} -> can be staffed by {', '.join(able)}")

Directory tree:
inventory.csv
volunteers.txt
work_orders
   WO-101.txt
   WO-102.txt
   WO-103.txt
   WO-104.txt
   WO-105.txt
   WO-106.txt
   WO-107.txt
   WO-108.txt
   WO-109.txt
   WO-110.txt
   WO-111.txt
   WO-112.txt
   WO-113.txt
   WO-114.txt
   WO-115.txt
   WO-116.txt
   WO-117.txt
   WO-118.txt
   WO-119.txt
   WO-120.txt
   WO-121.txt
   WO-122.txt
   WO-123.txt
   WO-124.txt
   WO-125.txt
   WO-126.txt
   WO-127.txt
   WO-128.txt
   WO-129.txt
   WO-130.txt
   WO-131.txt
   WO-132.txt
   WO-133.txt
   WO-134.txt
   WO-135.txt
   WO-136.txt
   WO-137.txt
   WO-138.txt
   WO-139.txt
   WO-140.txt

Sample work order (WO-101.txt):
Order ID: WO-101
Received: 2026-11-02
Customer: Hana
Bike: Cargo bike
Issue: Brake pads squeal but stop fine
Skill needed: brakes
Priority: low
Safety issue: no

inventory.csv:
part,on_hand,reorder_level
brake cable,2,5
disc brake pads,12,6
rim brake pads,9,4
inner tube 700c,3,10
inner tube 26in,14,6
chain 9-speed,1,4
quick release skewer,7,3
hydra

## Cell 6: Define the specialists and the helpers

**The three specialists.** These are the co-op's `AgentDefinition`s from earlier in the section, with one deliberate change: `shift-planner` now has the `Write` tool.

| Field | `parts-clerk` | `triage-mechanic` | `shift-planner` |
|---|---|---|---|
| Job | Find parts below their reorder level | Rank the urgent safety orders | Assign volunteers to given orders |
| `tools` | `Read`, `Glob`, `Grep` | `Read`, `Glob`, `Grep` | `Read`, `Glob`, `Grep`, **`Write`** |
| `model` | `MODEL_NAME` | `MODEL_NAME` | `MODEL_NAME` |

Why give the shift planner `Write`? In Cell 10 you block it from writing with a hook. A block only proves something if the write would otherwise have succeeded. With `Write` in the specialist's `tools` and in `allowed_tools`, nothing else stands in the way, so if the file is missing, the hook is the cause. The shift planner's system prompt tells it to save a plan when asked to.

**The helpers.** None of them orchestrates anything. They build options, run one prompt, and read the messages afterwards.

| Helper | What it does |
|---|---|
| `make_options(...)` | Builds one `ClaudeAgentOptions` with the specialists, `model=MODEL_NAME`, `cwd="/content"` and the `env` settings. Optional arguments add hooks, extra environment variables and other options such as `max_budget_usd`. |
| `run(prompt, **kwargs)` | Runs one `query()` with `make_options(**kwargs)` and returns every message in a list. |
| `show_result(msgs)` | Prints the final answer from the `ResultMessage`. |
| `timeline(msgs)` | Prints each specialist's start and finish in stream order, then the largest number of specialists that were running at the same moment. |

**`env` in every run.** `CLAUDE_CODE_DISABLE_BACKGROUND_TASKS` is always set to `"1"`, so the main agent waits for each specialist's report inside the same `query()`. Any `env` you pass to `make_options` is added on top. The Python SDK merges `env` into the inherited environment, so your `PATH` and API key are kept.

**`timeline()`** is the helper from the previous lecture, which treats a terminal status from either `TaskNotificationMessage` or `TaskUpdatedMessage` as finished. It has one addition for this notebook. When the concurrency cap refuses an Agent call, the call's result carries `Concurrent subagent limit reached`. The helper prints that call as `REFUSED` and leaves it out of the overlap count, because a refused specialist never ran.

In [38]:
import dataclasses
from claude_agent_sdk import (
    AgentDefinition, query, ClaudeAgentOptions, AssistantMessage, UserMessage,
    ResultMessage, ToolUseBlock, ToolResultBlock, TaskStartedMessage,
    TaskNotificationMessage, TaskUpdatedMessage, TERMINAL_TASK_STATUSES,
)

AGENTS = {
    "parts-clerk": AgentDefinition(
        description=(
            "Checks the co-op parts inventory and reports every part below its "
            "reorder level. Needs no input: it reads /content/bikecoop/inventory.csv."
        ),
        prompt=(
            "You are the Bike Repair Co-op's parts clerk. Read "
            "/content/bikecoop/inventory.csv and list every part whose on_hand is "
            "below its reorder_level, with both numbers. Keep the report short. "
            "Do not create or edit any files."
        ),
        tools=["Read", "Glob", "Grep"],
        model=MODEL_NAME,
    ),
    "triage-mechanic": AgentDefinition(
        description=(
            "Ranks the urgent work orders that have a safety issue. Needs no input: "
            "it reads the tickets in /content/bikecoop/work_orders/."
        ),
        prompt=(
            "You are the Bike Repair Co-op's triage mechanic. Read the work orders "
            "in /content/bikecoop/work_orders/. Find every order with "
            "'Priority: urgent' and 'Safety issue: yes'. Rank them from most to "
            "least dangerous to ride, one numbered line each: order ID, skill "
            "needed, and a one-sentence reason. Do not create or edit any files."
        ),
        tools=["Read", "Glob", "Grep"],
        model=MODEL_NAME,
    ),
    "shift-planner": AgentDefinition(
        description=(
            "Assigns volunteers to a given list of work orders. "
            "Needs the order IDs in its prompt."
        ),
        prompt=(
            "You are the Bike Repair Co-op's shift planner. You are given a list "
            "of work order IDs. Read only those orders in "
            "/content/bikecoop/work_orders/ and the volunteer list in "
            "/content/bikecoop/volunteers.txt. Assign each order to one volunteer "
            "whose skills include the skill the order needs, and pick one of that "
            "volunteer's shift days. If you are asked to save the plan, write it "
            "with your Write tool to the exact path you are given."
        ),
        tools=["Read", "Glob", "Grep", "Write"],
        model=MODEL_NAME,
    ),
}

AGENT_TOOL_NAMES = ("Agent", "Task")  # current and older name of the tool
BASE_TOOLS = ["Read", "Glob", "Grep", "Agent"]


def make_options(agents=None, tools=None, hooks=None, env=None, **extra):
    """One ClaudeAgentOptions for every run in this notebook."""
    return ClaudeAgentOptions(
        agents=agents or AGENTS,
        allowed_tools=tools or BASE_TOOLS,
        hooks=hooks,
        model=MODEL_NAME,
        cwd="/content",
        env={"CLAUDE_CODE_DISABLE_BACKGROUND_TASKS": "1", **(env or {})},
        **extra,
    )


async def run(prompt, **kwargs):
    """Run one query(); return every message from the stream."""
    msgs = []
    try:
        async for message in query(prompt=prompt, options=make_options(**kwargs)):
            msgs.append(message)
    except Exception as error:
        print(f"Run ended with an error: {error}")
    return msgs


def show_result(msgs):
    for m in msgs:
        if isinstance(m, ResultMessage):
            print(m.result)


def agent_calls(msgs):
    """Map each Agent call's id to the specialist it asked for."""
    calls = {}
    for m in msgs:
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock) and b.name in AGENT_TOOL_NAMES:
                    calls[b.id] = b.input.get("subagent_type", "general-purpose")
    return calls


def result_text(block):
    parts = block.content if isinstance(block.content, list) else [{"text": block.content or ""}]
    return "\n".join(p.get("text") or "" for p in parts)


def timeline(msgs):
    """Print each specialist's start and finish in stream order."""
    calls = agent_calls(msgs)
    owner, finished, refused, events = {}, set(), set(), []
    print("TIMELINE (stream order)")
    for m in msgs:
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock) and b.id in calls:
                    events.append(("start", b.id))
                    print(f"  START   {calls[b.id]:<16} Agent call sent")
        elif isinstance(m, TaskStartedMessage) and m.tool_use_id in calls:
            owner[m.task_id] = calls[m.tool_use_id]
            print(f"          {owner[m.task_id]:<16} task started")
        elif isinstance(m, (TaskNotificationMessage, TaskUpdatedMessage)):
            if m.status in TERMINAL_TASK_STATUSES and m.task_id not in finished:
                finished.add(m.task_id)
                name = owner.get(m.task_id) or calls.get(getattr(m, "tool_use_id", None), "?")
                print(f"          {name:<16} task {m.status}")
        elif isinstance(m, UserMessage) and isinstance(m.content, list):
            for b in m.content:
                if isinstance(b, ToolResultBlock) and b.tool_use_id in calls:
                    events.append(("end", b.tool_use_id))
                    if "limit reached" in result_text(b):
                        refused.add(b.tool_use_id)
                        print(f"  REFUSED {calls[b.tool_use_id]:<16} {result_text(b)[:60]}")
                    else:
                        print(f"  FINISH  {calls[b.tool_use_id]:<16} report returned")
    running, most_at_once = set(), 0
    for kind, call_id in events:
        if call_id in refused:
            continue
        if kind == "start":
            running.add(call_id)
            most_at_once = max(most_at_once, len(running))
        else:
            running.discard(call_id)
    print(f"Most specialists running at once: {most_at_once}")
    if refused:
        print(f"Agent calls refused by a limit: {len(refused)}")


for name, agent in AGENTS.items():
    print(f"{name:<16} tools={agent.tools}")

parts-clerk      tools=['Read', 'Glob', 'Grep']
triage-mechanic  tools=['Read', 'Glob', 'Grep']
shift-planner    tools=['Read', 'Glob', 'Grep', 'Write']


## Cell 7: Lifecycle log: SubagentStart and SubagentStop hooks

Two hook events fire around every subagent, whatever tools it uses:

| Event | When it fires | Fields this cell logs |
|---|---|---|
| `SubagentStart` | A subagent begins | `agent_type`, `agent_id` |
| `SubagentStop` | A subagent finishes | `agent_type`, `agent_id`, `agent_transcript_path` |

Both fields are **required** on these two events, so every entry has them. `agent_type` is the name from your `agents` dictionary, such as `parts-clerk`. `agent_id` is unique to one run of that specialist, so a start and its stop share the same `agent_id`.

Each callback has the standard hook signature `(input_data, tool_use_id, context)`. It appends one JSON line to `/content/bikecoop/agent_log.jsonl` and returns `{}`, which means "carry on, no changes". A lifecycle hook that only records never changes what the agent does.

`LIFECYCLE_HOOKS` is the dictionary you pass as `hooks=`. The `HookMatcher`s have no `matcher`, so they fire for every subagent. Writing to a file instead of printing gives you ground truth on disk that you can inspect after the run.

In [39]:
import json
from datetime import datetime
from claude_agent_sdk import HookMatcher

LOG = BASE / "agent_log.jsonl"


def log_line(entry):
    entry = {"time": datetime.now().strftime("%H:%M:%S.%f")[:-3], **entry}
    with open(LOG, "a") as f:
        f.write(json.dumps(entry) + "\n")


async def on_subagent_start(input_data, tool_use_id, context):
    log_line({
        "event": "SubagentStart",
        "agent_type": input_data["agent_type"],
        "agent_id": input_data["agent_id"],
    })
    return {}


async def on_subagent_stop(input_data, tool_use_id, context):
    log_line({
        "event": "SubagentStop",
        "agent_type": input_data["agent_type"],
        "agent_id": input_data["agent_id"],
        "agent_transcript_path": input_data["agent_transcript_path"],
    })
    return {}


LIFECYCLE_HOOKS = {
    "SubagentStart": [HookMatcher(hooks=[on_subagent_start])],
    "SubagentStop": [HookMatcher(hooks=[on_subagent_stop])],
}


def read_log():
    if not LOG.exists():
        return []
    return [json.loads(line) for line in LOG.read_text().splitlines()]


print("Lifecycle hooks ready. Log file:", LOG)

Lifecycle hooks ready. Log file: /content/bikecoop/agent_log.jsonl


## Cell 8: Run two specialists and read the log

This is the parallel request from the previous lecture: the parts clerk and the triage mechanic, called together. The only difference is `hooks=LIFECYCLE_HOOKS`.

The cell clears the log first, so everything in it comes from this run. After the run it prints:

1. The final answer, so you can see the work was done.
2. The `timeline()` of the run. Without a cap, the two specialists normally overlap. Cell 12 compares against this.
3. Every line of `agent_log.jsonl`.
4. A pairing check: for each `agent_id`, did a start **and** a stop arrive?

| What you should see | Why it matters |
|---|---|
| One `SubagentStart` and one `SubagentStop` per specialist | Every specialist that ran is accounted for |
| Matching `agent_id`s on each start and stop | You can tie the two events of one run together, even when specialists overlap |
| An `agent_transcript_path` on each stop | Cell 9 opens it |

The times and IDs differ on every run. The shape of the log does not.

In [40]:
LOG.unlink(missing_ok=True)

parallel_prompt = (
    "Two independent jobs for this week's co-op briefing. "
    "Use the parts-clerk agent to check stock, and the triage-mechanic agent "
    "to rank the urgent work orders. Neither job needs the other's result, "
    "so call both agents together in one step. "
    "Then combine their two reports into one short summary."
)

log_msgs = await run(parallel_prompt, hooks=LIFECYCLE_HOOKS)
show_result(log_msgs)
print()
timeline(log_msgs)

print("\nagent_log.jsonl:")
entries = read_log()
for e in entries:
    print(f"  {e['time']}  {e['event']:<14} {e['agent_type']:<16} {e['agent_id']}")

print("\nPairing check:")
for agent_id in dict.fromkeys(e["agent_id"] for e in entries):
    events = [e["event"] for e in entries if e["agent_id"] == agent_id]
    name = next(e["agent_type"] for e in entries if e["agent_id"] == agent_id)
    ok = "SubagentStart" in events and "SubagentStop" in events
    print(f"  [{'x' if ok else ' '}] {name:<16} {agent_id}: {', '.join(events)}")

**Co-op briefing, 2026-11-23**

**Low parts stock (4 items below reorder level):**
- Brake cable: 2 on hand, reorder at 5
- Inner tube 700c: 3 on hand, reorder at 10
- Chain 9-speed: 1 on hand, reorder at 4
- Hydraulic brake fluid: 1 on hand, reorder at 3

**Urgent safety work orders, most dangerous first:**
1. **WO-129 (frame):** crack next to the seat tube weld. A cracked frame can fail suddenly under load.
2. **WO-130 (brakes):** hydraulic fluid leaking onto the front rotor, so the main brake can fade or fail.
2. **WO-128 (wheels):** front quick release won't lock, so the wheel can come loose at speed. (The triage ranking puts this third; the order of WO-130 and WO-128 is close.)
4. **WO-103 (brakes):** frayed rear brake cable, so the rear brake is effectively gone. The front still works.
5. **WO-138 (drivetrain):** chain snapped and skips under load. Risk is lost drive or a jammed chain, not lost braking.

**Connection:** Brake cable and hydraulic brake fluid are both low, and WO-1

## Cell 9: Open a subagent's own transcript

Every subagent keeps its own transcript file. The `SubagentStop` entry in the log tells you where it is: `agent_transcript_path`.

This cell opens the transcript from the first stop entry and prints what that specialist did, one step per line:

| Step | What it is |
|---|---|
| First `TEXT` from `user` | The task the main agent handed to the specialist |
| `TOOL CALL` and `TOOL RESULT` | The specialist's own tool use and what each tool returned |
| Last `TEXT` from `assistant` | The specialist's final report, the only part the main agent receives |

**This is the subagent's own transcript, separate from the main session's.**

> The transcript format belongs to Claude Code rather than the SDK, so the cell reads it defensively and skips record types it does not recognise.

In [41]:
from pathlib import Path


def preview(value, width=70):
    if isinstance(value, list):  # tool results can be a list of text blocks
        value = " ".join(b.get("text", "") for b in value if isinstance(b, dict))
    text = value if isinstance(value, str) else json.dumps(value)
    text = " ".join(text.split())
    return text[:width] + ("..." if len(text) > width else "")


def transcript_steps(record):
    """Yield (label, detail) for each content block in one transcript record."""
    content = (record.get("message") or {}).get("content")
    if isinstance(content, str):
        yield "TEXT", preview(content)
        return
    for block in content or []:
        kind = block.get("type")
        if kind == "text":
            yield "TEXT", preview(block.get("text", ""))
        elif kind == "tool_use":
            yield "TOOL CALL", f"{block.get('name')} {preview(block.get('input', {}), 55)}"
        elif kind == "tool_result":
            yield "TOOL RESULT", preview(block.get("content", ""))


stops = [e for e in read_log() if e["event"] == "SubagentStop"]
if not stops:
    print("No SubagentStop entry in the log. Run Cell 8 first.")
else:
    first = stops[0]
    path = Path(first["agent_transcript_path"]).expanduser()
    print("Specialist:", first["agent_type"])
    print("Transcript:", path)
    if not path.exists():
        print("The transcript file is not on disk.")
    else:
        records = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
        print(f"\nWHAT {first['agent_type'].upper()} DID, STEP BY STEP")
        step = 0
        for r in records:
            if r.get("type") not in ("user", "assistant"):
                continue
            for label, detail in transcript_steps(r):
                step += 1
                print(f"{step:>3}  {r['type']:<10} {label:<12} {detail}")

Specialist: parts-clerk
Transcript: /root/.claude/projects/-content/b6123b81-98f1-4be0-88e6-a55c4e3f7248/subagents/agent-a1c434a07bbbc10f3.jsonl

WHAT PARTS-CLERK DID, STEP BY STEP
  1  user       TEXT         Check the co-op parts inventory at /content/bikecoop/inventory.csv and...
  2  assistant  TOOL CALL    Read {"file_path": "/content/bikecoop/inventory.csv"}
  3  user       TOOL RESULT  1 part,on_hand,reorder_level 2 brake cable,2,5 3 disc brake pads,12,6 ...
  4  assistant  TEXT         Four parts are below their reorder level in /content/bikecoop/inventor...


## Cell 10: Block one specialist with a PreToolUse hook

Hooks are the **precise** tool: they can tell subagents apart. `PreToolUse`, `PostToolUse`, `PostToolUseFailure` and `PermissionRequest` inputs carry two optional fields:

| Field | Inside a subagent | On the main thread |
|---|---|---|
| `agent_type` | The specialist's name, such as `shift-planner` | Absent |
| `agent_id` | Matches the `agent_id` in that subagent's `SubagentStart` and `SubagentStop` | Absent |

When specialists run in parallel, their tool hooks arrive interleaved. These fields are the only reliable way to know which subagent a tool call came from.

This hook is registered with `matcher="Write"`, so it fires only for `Write` calls. It reads `input_data.get("agent_type")`:

- **`shift-planner`**: deny. It returns `permissionDecision: "deny"` inside `hookSpecificOutput`, with a `permissionDecisionReason` that the agent reads as the tool result.
- **Anything else, including the main agent**: allow, by returning `{}`.

`.get()` is used because the field is absent on the main thread. Every decision is printed and kept in `WRITE_DECISIONS`, with `main` standing in for the main agent.

> When several `PreToolUse` hooks fire for one call, they run in parallel and the most restrictive decision wins. One `deny` is enough to block the call.

In [42]:
WRITE_DECISIONS = []


async def block_shift_planner_writes(input_data, tool_use_id, context):
    who = input_data.get("agent_type") or "main"
    target = input_data["tool_input"].get("file_path", "")
    if input_data.get("agent_type") == "shift-planner":
        WRITE_DECISIONS.append((who, "DENY", target))
        print(f"[hook] DENY  Write by {who}: {target}")
        return {
            "hookSpecificOutput": {
                "hookEventName": "PreToolUse",
                "permissionDecision": "deny",
                "permissionDecisionReason": (
                    "The shift-planner may plan, but it may not save files. "
                    "Return the plan as text instead."
                ),
            }
        }
    WRITE_DECISIONS.append((who, "allow", target))
    print(f"[hook] allow Write by {who}: {target}")
    return {}


WRITE_GUARD = {"PreToolUse": [HookMatcher(matcher="Write", hooks=[block_shift_planner_writes])]}
print("Write guard ready.")

Write guard ready.


## Cell 11: Run it: one write blocked, one allowed

Now the guard meets a real run. The prompt asks for two writes:

| Who | Asked to write | Expected on disk |
|---|---|---|
| `shift-planner` | `/content/bikecoop/rota.txt`, its staffing plan | **Missing**: the hook denies it |
| The main agent | `/content/bikecoop/summary.txt`, two lines | **Present**: the hook allows it |

`Write` is in `allowed_tools` and in the shift planner's `tools`, so without the hook both writes would succeed. That is what makes the result attributable: if `rota.txt` is missing, the hook is the only cause.

The prompt also tells the main agent not to write `rota.txt` itself. Without that line, the main agent could quietly save the plan on the specialist's behalf, and the file check would no longer show the block.

The hooks dictionary combines the write guard with the lifecycle hooks from Cell 7, so the log shows the shift planner starting and stopping too. After the run, the cell prints the files on disk and the hook's decision log.

In [43]:
for name in ("rota.txt", "summary.txt"):
    (BASE / name).unlink(missing_ok=True)
LOG.unlink(missing_ok=True)
WRITE_DECISIONS.clear()

rota_prompt = (
    "Use the shift-planner agent to staff these urgent work orders: "
    f"{', '.join(URGENT_IDS)}. Tell it to save its plan with its own Write tool "
    "to /content/bikecoop/rota.txt. Only the shift-planner may write rota.txt. "
    "If it cannot, do not write rota.txt yourself: just report what happened. "
    "Then write /content/bikecoop/summary.txt yourself, with exactly two lines: "
    "how many orders were planned, and whether rota.txt was saved."
)

rota_msgs = await run(
    rota_prompt,
    tools=["Read", "Glob", "Grep", "Write", "Agent"],
    hooks={**LIFECYCLE_HOOKS, **WRITE_GUARD},
)
show_result(rota_msgs)

print("\nFiles on disk:")
for name in ("rota.txt", "summary.txt"):
    print(f"  {name:<12} exists: {(BASE / name).exists()}")
if (BASE / "summary.txt").exists():
    print("\nsummary.txt:\n" + (BASE / "summary.txt").read_text())

print("Hook decision log:")
for who, decision, target in WRITE_DECISIONS:
    print(f"  {decision:<5} {who:<14} {target}")
print("\nSpecialists in agent_log.jsonl:", [e["agent_type"] for e in read_log() if e["event"] == "SubagentStart"])

[hook] DENY  Write by shift-planner: /content/bikecoop/rota.txt
[hook] allow Write by main: /content/bikecoop/summary.txt
The shift-planner planned all five work orders, but rota.txt was not saved. A PreToolUse hook rejected its Write call to /content/bikecoop/rota.txt, with the message "The shift-planner may plan, but it may not save files. Return the plan as text instead." Per your instructions, I did not write rota.txt myself.

I wrote /content/bikecoop/summary.txt with these two lines:

```
Orders planned: 5
rota.txt saved: no
```

The plan the shift-planner returned, which is not saved to any file:

| Work order | Skill needed | Volunteer | Shift |
|---|---|---|---|
| WO-103 | brakes | Asha | Tue |
| WO-128 | wheels | Dev | Fri |
| WO-129 | frame | Elena | Thu |
| WO-130 | brakes | Carmen | Sat |
| WO-138 | drivetrain | Ben | Mon |

Asha and Carmen both have brakes, so WO-103 and WO-130 are split between them. To get the rota on disk, the hook block on the shift-planner's Write ne

## Cell 12: Concurrency cap: one specialist at a time

The limits in `env` are the **blunt** tool. They apply to every subagent in the query, whatever its name.

| Environment variable | Default | At the limit |
|---|---|---|
| `CLAUDE_CODE_MAX_CONCURRENT_SUBAGENTS` | `20` subagents running at once | Claude Code refuses to start another subagent until the running count drops below the limit. The refused Agent call returns `Concurrent subagent limit reached`. |
| `CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH` | `3` layers below the main agent | A subagent at the bottom layer cannot start subagents of its own (Cell 13) |

Both are environment variables for the Claude Code process. You pass them as **strings** through `env=`, and the Python SDK merges them into the environment. They need Python SDK 0.2.127 or later.

This cell reruns the parallel request from Cell 8 with the concurrency cap set to `"1"`. The prompt still asks for both specialists together. Watch the timeline:

- In Cell 8, without a cap, *Most specialists running at once* was normally 2.
- With the cap, only one specialist can run at a time. If the main agent asks for both at once, the second call shows as `REFUSED`, and the main agent has to call it again after the first finishes, or do that job itself.

Either way, the timeline should show the specialists no longer overlapping. The cap does not queue the second call for you. It refuses it, and the main agent decides what to do next.

In [44]:
capped_msgs = await run(
    parallel_prompt,
    env={"CLAUDE_CODE_MAX_CONCURRENT_SUBAGENTS": "1"},
)
show_result(capped_msgs)
print()
timeline(capped_msgs)

The parts check finished, but the triage-mechanic agent did not run. The system refused it because the concurrent subagent limit is one, and it said not to retry. I'm not retrying it, so the urgent work order ranking is still missing.

**Parts below reorder level** (from `/content/bikecoop/inventory.csv`):

| Part | On hand | Reorder level |
|---|---|---|
| Brake cable | 2 | 5 |
| Inner tube 700c | 3 | 10 |
| Chain 9-speed | 1 | 4 |
| Hydraulic brake fluid | 1 | 3 |

**Urgent safety work orders:** not ranked yet. To get this, either raise `CLAUDE_CODE_MAX_CONCURRENT_SUBAGENTS` so both agents can run together, or I can run the triage-mechanic agent by itself now that the parts check is done.

TIMELINE (stream order)
  START   parts-clerk      Agent call sent
          parts-clerk      task started
  START   triage-mechanic  Agent call sent
  REFUSED triage-mechanic  Concurrent subagent limit reached. You can run 1 subagents a
          parts-clerk      task completed
  FINISH  parts-cle

## Cell 13: Depth cap: no subagents inside subagents

A subagent that has the `Agent` tool can start subagents of its own, so one prompt can grow into a tree. `CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH` sets how many layers that tree may have below the main agent. At `"1"`, your specialists cannot start any subagents of their own.

To test that, this cell makes a copy of the specialists in which `triage-mechanic` **does** have `Agent` in its `tools`. `dataclasses.replace` copies an `AgentDefinition` with one field changed, so `AGENTS` itself is untouched. Then the prompt asks the triage mechanic to hand half the reading to another subagent.

| Check | Where it comes from | Expected with depth 1 |
|---|---|---|
| Agent calls with a `parent_tool_use_id` | The message stream: an Agent call made from inside a subagent | 0 |
| `SubagentStart` entries in the log | The lifecycle hooks from Cell 7 | 1, the triage mechanic only |
| Urgent orders found | The final answer compared with `URGENT_IDS` | All 5, because the specialist does the reading itself |

The documented behaviour at the limit is that the bottom-layer subagent is unable to spawn, so it does its delegated work itself. You won't necessarily see an error message. The evidence is the zero and the one.

In [45]:
NESTING_AGENTS = {
    **AGENTS,
    "triage-mechanic": dataclasses.replace(
        AGENTS["triage-mechanic"],
        tools=["Read", "Glob", "Grep", "Agent"],
    ),
}
LOG.unlink(missing_ok=True)

depth_prompt = (
    "Use the triage-mechanic agent to rank the urgent safety orders. "
    "Tell it to split the reading: it must hand work orders WO-101 to WO-120 "
    "to another subagent of its own, and read WO-121 to WO-140 itself."
)

depth_msgs = await run(
    depth_prompt,
    agents=NESTING_AGENTS,
    hooks=LIFECYCLE_HOOKS,
    env={"CLAUDE_CODE_MAX_SUBAGENT_SPAWN_DEPTH": "1"},
)
show_result(depth_msgs)

nested_calls = [
    b for m in depth_msgs
    if isinstance(m, AssistantMessage) and m.parent_tool_use_id
    for b in m.content
    if isinstance(b, ToolUseBlock) and b.name in AGENT_TOOL_NAMES
]
starts = [e["agent_type"] for e in read_log() if e["event"] == "SubagentStart"]
answer = next((m.result or "" for m in depth_msgs if isinstance(m, ResultMessage)), "")

print("\nAgent calls made from inside a subagent:", len(nested_calls))
print("SubagentStart entries in the log:        ", len(starts), starts)
print("Urgent orders in the final answer:       ",
      f"{sum(oid in answer for oid in URGENT_IDS)} of {len(URGENT_IDS)}")

The triage-mechanic returned a ranked list of five urgent safety work orders, but it did not follow the split you asked for. It said the Agent tool was not available to it, so it read all of WO-101 to WO-140 itself. I have not checked its ticket readings against the files.

Ranked most to least dangerous to ride:

1. **WO-128** (wheels, Grace, folding bike, received 2026-11-15): the front quick release won't lock, so the front wheel can come loose from the fork at speed.
2. **WO-129** (frame, Tom, mountain bike, received 2026-11-02): a crack next to the seat tube weld. It is usually not safely repairable, so the frame may need retiring. It is the oldest open urgent ticket.
3. **WO-130** (brakes, Kwame, road bike, received 2026-11-03): hydraulic fluid leaking onto the front rotor, which risks sudden loss of the main stopping brake.
4. **WO-138** (drivetrain, Lena, road bike, received 2026-11-11): the chain snapped once on a hill and now skips under load, which can cause a sudden loss of

## Cell 14: Spending cap: max_budget_usd and ResultError

`max_budget_usd` is an option on `ClaudeAgentOptions`. It is compared against the query's estimated spend, `total_cost_usd`, which **includes** subagent requests. At the cap, Claude Code refuses to start more subagents and ends the query with the `ResultMessage` subtype `"error_max_budget_usd"`.

With a single `query()`, the stream yields that `ResultMessage` first, and then the SDK raises **`ResultError`**. It is a subclass of `ProcessError`, imported from `claude_agent_sdk`, and it needs SDK 0.2.140 or later. Its attributes carry the result's fields, so you can branch on why the run ended without parsing text:

| Attribute | What it holds |
|---|---|
| `subtype` | The result subtype, here `error_max_budget_usd` |
| `terminal_reason` | Why the loop ended, as reported by Claude Code. For a budget stop it is `budget_exhausted` |
| `session_id` | The session the run belonged to |
| `errors`, `result`, `api_error_status`, `data` | Error strings, result text, HTTP status of a failed API call, and the raw result payload |

This cell runs the full briefing pipeline from the previous lecture with a deliberately tiny cap: `0.001`. It writes its own `async for` loop, inside `try`/`except ResultError as e`, so you can see the exception directly. It also keeps every message in `budget_msgs`, so afterwards `agent_calls()` can list the specialists that were started before the stop.

**Budget headroom.** The spend is checked after each model response arrives. The response that crosses the cap still completes and counts, so `total_cost_usd` can end up above the cap. With a cap this small, the run usually stops right after the main agent's first response. If that response already contained the Agent calls, those specialists have started. That still demonstrates the cap. The list of specialists that started and the missing `briefing.md` show how far the run got. A specialist that had already started may carry on for a moment after the cap.

**No hooks in this cell.** In SDK 0.2.174, a `query()` with hooks that ends on a budget stop can keep waiting indefinitely, so the evidence here comes from the message stream instead of the lifecycle log.

All dollar figures here are client-side estimates computed by the SDK, not a bill.

In [46]:
from claude_agent_sdk import ResultError

briefing = BASE / "briefing.md"
briefing.unlink(missing_ok=True)

pipeline_prompt = """Prepare this week's Bike Repair Co-op briefing.

Step 1: Use the parts-clerk agent and the triage-mechanic agent. Their jobs are
independent, so call both agents together in one step.
Step 2: When both reports are back, use the shift-planner agent to staff the five
urgent orders from the triage report. Write the five order IDs into its prompt,
because it cannot see the triage report.
Step 3: Write /content/bikecoop/briefing.md yourself, with exactly three sections:
## Low stock, ## Urgent orders, ## Assignments.

Do not let subagents create files. Only you write briefing.md."""

budget_msgs = []
try:
    async for message in query(
        prompt=pipeline_prompt,
        options=make_options(
            tools=["Read", "Glob", "Grep", "Agent", "Write"],
            max_budget_usd=0.001,
        ),
    ):
        budget_msgs.append(message)
        if isinstance(message, ResultMessage):
            print(f"ResultMessage: subtype={message.subtype}  "
                  f"estimated cost=${message.total_cost_usd or 0:.4f}")
    print("The run finished without hitting the cap.")
except ResultError as e:
    print("ResultError raised")
    print("  subtype:        ", e.subtype)
    print("  terminal_reason:", e.terminal_reason)
    print("  session_id:     ", e.session_id)

print("\nSpecialists started before the stop:", list(agent_calls(budget_msgs).values()))
print("briefing.md exists:", briefing.exists())

ResultMessage: subtype=error_max_budget_usd  estimated cost=$0.0011
ResultError raised
  subtype:         error_max_budget_usd
  terminal_reason: budget_exhausted
  session_id:      5b70cd3e-611f-4cde-bf73-8e12ab2b76cb

Specialists started before the stop: ['parts-clerk', 'triage-mechanic']
briefing.md exists: False


## Cell 15: Where subagent cost shows up

The `ResultMessage` reports usage in three places, and they don't all count the same work:

| Field | Counts subagents? | What it is |
|---|---|---|
| `usage` | **No**. Main agent loop only | A dict of token counts: `input_tokens`, `output_tokens`, `cache_read_input_tokens`, `cache_creation_input_tokens` |
| `total_cost_usd` | Yes | The SDK's estimated cost for the whole query |
| `model_usage` | Yes | A dict keyed by model name. Each entry has `inputTokens`, `outputTokens`, `cacheReadInputTokens`, `cacheCreationInputTokens` and `costUSD` |

So if you price `usage` yourself, you get roughly the main agent's share. The gap between that and `total_cost_usd` is roughly what the specialists cost.

**The rates.** `RATES` holds Claude Haiku 5.5 rates from the pricing page, in USD per million tokens, for prompts up to 100,000 tokens. Check https://platform.claude.com/docs/en/about-claude/pricing for current rates before you rely on them.

| Token type | Prompts up to 100,000 tokens | Prompts over 100,000 tokens |
|---|---|---|
| Input | 0.10 | 0.50 |
| Output | 0.50 | 2.50 |
| 5-minute cache write | 0.125 | 0.625 |
| 1-hour cache write | 0.20 | 1.00 |
| Cache read | 0.01 | 0.05 |

Each request is priced on its own, and its prompt length includes cache reads and writes. The requests in this notebook stay well under 100,000 tokens, so the cell uses the first column. With an API key, cache writes default to the 5-minute lifetime, so the cell prices `cache_creation_input_tokens` at the 5-minute rate.

**Estimates, not a bill.** `total_cost_usd` and `costUSD` are computed by the SDK from a price table bundled with it. For what you are actually billed, use the Usage page in the Claude Console.

In [47]:
# Claude Haiku 5.5, USD per million tokens, prompts up to 100,000 tokens.
# Source: https://platform.claude.com/docs/en/about-claude/pricing
RATES = {
    "input": 0.10,
    "output": 0.50,
    "cache_write_5m": 0.125,
    "cache_read": 0.01,
}

cost_msgs = await run(parallel_prompt)
result = next(m for m in cost_msgs if isinstance(m, ResultMessage))
u = result.usage or {}

rows = [
    ("input", u.get("input_tokens", 0), RATES["input"]),
    ("output", u.get("output_tokens", 0), RATES["output"]),
    ("cache read", u.get("cache_read_input_tokens", 0), RATES["cache_read"]),
    ("cache write", u.get("cache_creation_input_tokens", 0), RATES["cache_write_5m"]),
]
print("MAIN AGENT ONLY (ResultMessage.usage)")
print(f"  {'token type':<12}{'tokens':>10}{'est. $':>12}")
main_estimate = 0.0
for label, tokens, rate in rows:
    dollars = tokens * rate / 1_000_000
    main_estimate += dollars
    print(f"  {label:<12}{tokens:>10,}{dollars:>12.5f}")
print(f"  {'total':<12}{'':>10}{main_estimate:>12.5f}")

total = result.total_cost_usd or 0
print("\nWHOLE QUERY, SUBAGENTS INCLUDED")
print(f"  total_cost_usd:                 {total:.5f}")
print(f"  gap (roughly the specialists):  {total - main_estimate:.5f}")

print("\nmodel_usage (per model, subagents included)")
for model, mu in (result.model_usage or {}).items():
    print(f"  {model}: in={mu['inputTokens']:,} out={mu['outputTokens']:,} "
          f"cache_read={mu['cacheReadInputTokens']:,} "
          f"cache_write={mu['cacheCreationInputTokens']:,}  est. ${mu['costUSD']:.5f}")

MAIN AGENT ONLY (ResultMessage.usage)
  token type      tokens      est. $
  input                4     0.00000
  output             996     0.00050
  cache read      31,724     0.00032
  cache write      1,484     0.00019
  total                      0.00100

WHOLE QUERY, SUBAGENTS INCLUDED
  total_cost_usd:                 0.00324
  gap (roughly the specialists):  0.00224

model_usage (per model, subagents included)
  claude-haiku-5-5: in=1,271 out=3,689 cache_read=54,450 cache_write=5,824  est. $0.00324
